In [ ]:
from pyspark.sql import SparkSession
from pyspark import SparkContext, SQLContext
from pyspark.sql.functions import when, rand, udf, expr, floor, col
from pyspark.sql.types import StringType
import random

sc = SparkContext.getOrCreate()
sqlContext = SQLContext(sc)
spark = SparkSession.builder.master("local[*]").appName("Exercicio Spark").getOrCreate()

/usr/local/lib/python3.11/dist-packages/pyspark/sql/context.py:113: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(


In [ ]:
df_nomes= spark.read.csv("/content/drive/MyDrive/Colab Notebooks/nomes_aleatorios.txt")

df_nomes.show(5)

+----------------+
|             _c0|
+----------------+
|  Frances Bennet|
|   Jamie Russell|
|  Edward Kistler|
|   Sheila Maurer|
|Donald Golightly|
+----------------+
only showing top 5 rows



In [ ]:
df_nomes = df_nomes.withColumnRenamed("_c0", "Nomes")
df_nomes.printSchema()

df_nomes.show(10)

root
 |-- Nomes: string (nullable = true)

+-----------------+
|            Nomes|
+-----------------+
|   Frances Bennet|
|    Jamie Russell|
|   Edward Kistler|
|    Sheila Maurer|
| Donald Golightly|
|       David Gray|
|      Joy Bennett|
|      Paul Kriese|
|Berniece Ornellas|
|    Brian Farrell|
+-----------------+
only showing top 10 rows



In [ ]:
df_nomes = df_nomes.withColumn("escolaridade",when(rand() < 0.33, "Fundamental").when(rand() < 0.66, "Médio").otherwise("Superior"))

df_nomes.show(10)

+-----------------+------------+
|            Nomes|escolaridade|
+-----------------+------------+
|   Frances Bennet|       Médio|
|    Jamie Russell|       Médio|
|   Edward Kistler|       Médio|
|    Sheila Maurer|    Superior|
| Donald Golightly|    Superior|
|       David Gray|    Superior|
|      Joy Bennett| Fundamental|
|      Paul Kriese|       Médio|
|Berniece Ornellas|    Superior|
|    Brian Farrell|       Médio|
+-----------------+------------+
only showing top 10 rows



In [ ]:
paises = ["Venezuela", "Uruguai", "Suriname", "Guiana", "Peru", "Paraguai", "Chile", "Colombia", "Equador", "Bolivia", "Brasil", "Argentina"]

def paisaleatorio():
  return random.choice(paises)

pais_aleatorio = udf(paisaleatorio, StringType())
df_nomes = df_nomes.withColumn("Pais", pais_aleatorio())

df_nomes.show(10)

+-----------------+------------+---------+
|            Nomes|escolaridade|     Pais|
+-----------------+------------+---------+
|   Frances Bennet|       Médio|   Guiana|
|    Jamie Russell|       Médio|   Guiana|
|   Edward Kistler|       Médio|Argentina|
|    Sheila Maurer|    Superior|     Peru|
| Donald Golightly|    Superior|   Brasil|
|       David Gray|    Superior| Suriname|
|      Joy Bennett| Fundamental|Venezuela|
|      Paul Kriese|       Médio| Colombia|
|Berniece Ornellas|    Superior|   Brasil|
|    Brian Farrell|       Médio|Argentina|
+-----------------+------------+---------+
only showing top 10 rows



In [ ]:
df_nomes = df_nomes.withColumn("AnoNascimento", (floor(rand() * (2010 - 1945 + 1)) + 1945))

df_nomes.show(10)

+-----------------+------------+---------+-------------+
|            Nomes|escolaridade|     Pais|AnoNascimento|
+-----------------+------------+---------+-------------+
|   Frances Bennet|       Médio| Paraguai|         2002|
|    Jamie Russell|       Médio| Colombia|         1988|
|   Edward Kistler|       Médio|  Uruguai|         1994|
|    Sheila Maurer|    Superior|Venezuela|         1980|
| Donald Golightly|    Superior|    Chile|         1959|
|       David Gray|    Superior| Paraguai|         1987|
|      Joy Bennett| Fundamental|   Brasil|         1998|
|      Paul Kriese|       Médio|    Chile|         1994|
|Berniece Ornellas|    Superior|  Uruguai|         1946|
|    Brian Farrell|       Médio|  Equador|         1992|
+-----------------+------------+---------+-------------+
only showing top 10 rows



In [ ]:
df_select = df_nomes.filter(col("AnoNascimento") >= 2000)

df_select.show(10)

+--------------------+------------+---------+-------------+
|               Nomes|escolaridade|     Pais|AnoNascimento|
+--------------------+------------+---------+-------------+
|      Frances Bennet|       Médio| Paraguai|         2002|
|       Tracy Herring| Fundamental|  Bolivia|         2002|
|        David Medina|       Médio|   Guiana|         2001|
|        Rebecca Snow|       Médio| Paraguai|         2009|
|      Gabriel Colyer| Fundamental| Paraguai|         2000|
|        Roxie Bernal| Fundamental|  Equador|         2004|
|       Michael Agnew|    Superior|Argentina|         2000|
|Christopher Williams|       Médio|Venezuela|         2000|
|        Juliet Liles|    Superior| Colombia|         2006|
|    Richelle Vasquez|    Superior|  Equador|         2009|
+--------------------+------------+---------+-------------+
only showing top 10 rows



In [ ]:
df_nomes.createOrReplaceTempView("pessoas")

spark.sql("SELECT * FROM pessoas WHERE AnoNascimento >= 2000").show()

+--------------------+------------+---------+-------------+
|               Nomes|escolaridade|     Pais|AnoNascimento|
+--------------------+------------+---------+-------------+
|      Frances Bennet|       Médio|     Peru|         2002|
|       Tracy Herring| Fundamental|Argentina|         2002|
|        David Medina|       Médio|   Guiana|         2001|
|        Rebecca Snow|       Médio|   Brasil|         2009|
|      Gabriel Colyer| Fundamental|   Brasil|         2000|
|        Roxie Bernal| Fundamental|  Uruguai|         2004|
|       Michael Agnew|    Superior|  Equador|         2000|
|Christopher Williams|       Médio|   Guiana|         2000|
|        Juliet Liles|    Superior| Suriname|         2006|
|    Richelle Vasquez|    Superior|  Uruguai|         2009|
|        Douglas Boyd|       Médio|  Uruguai|         2007|
|      Robert Andrews|    Superior| Suriname|         2004|
|     Kenneth Rayburn|       Médio|  Bolivia|         2009|
|        Daniel Reese| Fundamental|  Equ

In [ ]:
millennials_df = df_nomes.filter((col("AnoNascimento") >= 1980) & (col("AnoNascimento") <= 1994))
millennials_df.show()

millennials_count = millennials_df.count()
print(f"Número de pessoas da geração Millennials: {millennials_count}")

+----------------+------------+---------+-------------+
|           Nomes|escolaridade|     Pais|AnoNascimento|
+----------------+------------+---------+-------------+
|   Jamie Russell|       Médio|  Bolivia|         1988|
|  Edward Kistler|       Médio|Argentina|         1994|
|   Sheila Maurer|    Superior|Venezuela|         1980|
|      David Gray|    Superior| Suriname|         1987|
|     Paul Kriese|       Médio| Paraguai|         1994|
|   Brian Farrell|       Médio|   Guiana|         1992|
|Wallace Mitchell|       Médio|  Bolivia|         1983|
|        Mary Lee|    Superior| Suriname|         1985|
|  Wilfredo Grant| Fundamental| Paraguai|         1983|
|      John Meyer|    Superior|   Guiana|         1980|
|   Ricky Gilbert|    Superior|   Guiana|         1990|
|     Lisa Baxley|    Superior|     Peru|         1984|
|Cristina Sheston|    Superior|    Chile|         1994|
|      April Ward| Fundamental|  Uruguai|         1985|
| Katherine Moore| Fundamental|Venezuela|       

In [ ]:
millennials_df = spark.sql("SELECT * FROM pessoas WHERE AnoNascimento >= 1980 AND AnoNascimento <= 1994")
millennials_df.show()


millennials_count = millennials_df.count()
print(f"Número de pessoas da geração Millennials: {millennials_count}")

+----------------+------------+--------+-------------+
|           Nomes|escolaridade|    Pais|AnoNascimento|
+----------------+------------+--------+-------------+
|   Jamie Russell|       Médio|Colombia|         1988|
|  Edward Kistler|       Médio|  Guiana|         1994|
|   Sheila Maurer|    Superior|  Guiana|         1980|
|      David Gray|    Superior| Uruguai|         1987|
|     Paul Kriese|       Médio| Equador|         1994|
|   Brian Farrell|       Médio|   Chile|         1992|
|Wallace Mitchell|       Médio| Bolivia|         1983|
|        Mary Lee|    Superior| Uruguai|         1985|
|  Wilfredo Grant| Fundamental| Equador|         1983|
|      John Meyer|    Superior| Uruguai|         1980|
|   Ricky Gilbert|    Superior|Paraguai|         1990|
|     Lisa Baxley|    Superior|  Brasil|         1984|
|Cristina Sheston|    Superior|    Peru|         1994|
|      April Ward| Fundamental|  Brasil|         1985|
| Katherine Moore| Fundamental| Uruguai|         1994|
|   Sherry

In [ ]:
query = """
SELECT
    pais,
    CASE
        WHEN AnoNascimento BETWEEN 1964 AND 1946 THEN 'Baby Boomer'
        WHEN AnoNascimento BETWEEN 1965 AND 1979 THEN 'Geração X'
        WHEN AnoNascimento BETWEEN 1980 AND 1994 THEN 'Geração Y'
        WHEN AnoNascimento BETWEEN 1995 AND 2015 THEN 'Geração Z'
        ELSE 'Outras Gerações'
    END AS geracao,
    COUNT(*) AS quantidade
FROM pessoas
GROUP BY pais, geracao
ORDER BY pais, geracao, quantidade
"""
df_resultado = spark.sql(query)

df_resultado.show()


+---------+---------------+----------+
|     pais|        geracao|quantidade|
+---------+---------------+----------+
|Argentina|      Geração X|    189178|
|Argentina|      Geração Y|    189360|
|Argentina|      Geração Z|    202247|
|Argentina|Outras Gerações|    252454|
|  Bolivia|      Geração X|    189913|
|  Bolivia|      Geração Y|    189361|
|  Bolivia|      Geração Z|    201720|
|  Bolivia|Outras Gerações|    251389|
|   Brasil|      Geração X|    189435|
|   Brasil|      Geração Y|    188798|
|   Brasil|      Geração Z|    201634|
|   Brasil|Outras Gerações|    252458|
|    Chile|      Geração X|    190049|
|    Chile|      Geração Y|    189251|
|    Chile|      Geração Z|    201918|
|    Chile|Outras Gerações|    252078|
| Colombia|      Geração X|    189400|
| Colombia|      Geração Y|    189618|
| Colombia|      Geração Z|    202300|
| Colombia|Outras Gerações|    252484|
+---------+---------------+----------+
only showing top 20 rows

